<a href="https://colab.research.google.com/github/fataa34/applied-optimization-with-python/blob/unconstraint/20251020_Praktikum_MO_2_Unconstraint_1_Dimensi.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Permasalahan Optimasi Satu Dimensi (1D)

Permasalahan optimasi satu dimensi (1D) melibatkan pencarian nilai optimal (maksimum atau minimum) dari sebuah fungsi yang hanya bergantung pada satu variabel. Dalam konteks mencari minimum, tujuannya adalah menemukan nilai `x` yang membuat fungsi `f(x)` mencapai nilai terkecilnya.


DEFINISI MASALAH

Untuk menyambungkan dengan pembahasan selanjutnya, variabel satu dimensi akan diidentifikasi sebagai **α** alih-alih **x** atau **x₁**.  
Dalam bab-bab berikutnya, ini juga disebut sebagai *perhitungan langkah satu dimensi (one-dimensional stepsize computation)*.

Contoh:

**Minimalkan:**

$$
f(\alpha) = (\alpha - 1)^2 (\alpha - 2)(\alpha - 3)
$$

**Dengan batasan:**

$$
0 \leq \alpha \leq 4
$$

Masalah ini tidak merepresentasikan rancangan nyata apa pun.  
Masalah ini dibuat agar memiliki beberapa titik minimum lokal dalam area yang diminati.  
Persamaan diatas merepresentasikan **masalah tanpa batasan (unconstrained problem)**.  
Batasan samping (side constraints) disertakan pada semua masalah di buku ini untuk menyampaikan ide bahwa sebenarnya tidak ada masalah yang benar-benar tanpa batasan.  
Batasan tersebut juga mendefinisikan wilayah desain yang dapat diterima untuk semua masalah.

## Metode Newton

### Algoritma Umum Metode Newton Raphson

**Langkah-langkah:**

1. Asumsikan nilai awal **α**
2. Hitung **Δα**
3. Perbarui nilai **α̃ = α + Δα**

   - Jika telah konvergen (φ(α̃) = 0), maka **keluar**
   - Jika belum konvergen (φ(α̃) ≠ 0), maka set **α ← α̃** dan **kembali ke Langkah 2**

---
dimana :

$$
\phi(\tilde{\alpha}) = \phi(\alpha + \Delta\alpha)
= \phi(\alpha) + \frac{d\phi}{d\alpha}\Delta\alpha = 0
$$

$$
\Delta\alpha = -\frac{\phi(\alpha)}{\frac{d\phi}{d\alpha}}
= -\left[\frac{d\phi}{d\alpha}\right]^{-1}\phi(\alpha)
$$

---

Persamaan diatas menunjukkan bahwa perubahan langkah (**Δα**) dapat dihitung dari nilai fungsi **φ(α)** dan turunannya.  
Algoritma ini pada dasarnya menggambarkan **metode Newton-Raphson** untuk mencari akar dari φ(α).


In [ ]:
import numpy as np
import pandas as pd

#definisikan fungsi objektif nya
def fungsi_objektif(x):
  return (x-1)**2*(x-2)*(x-3)

#definisikan turunan pertama fungsi objektif
def turunan_pertama(x):
  return 4*x**3-21*x**2+34*x-17

#definisikan turunan kedua fungsi obkektif
def turunan_kedua(x):
  return 12*x**2-42*x+34

In [ ]:
def newton_raphson(x0, epsilon, max_iter):
  x = x0
  results = []
  for i in range(max_iter):
    f_x = fungsi_objektif(x)
    f_prime = turunan_pertama(x)
    f_double_prime = turunan_kedua(x)
    delta_x = -f_prime / f_double_prime
    x_new = x + delta_x
    sigma_x = turunan_pertama(x_new)
    if abs(sigma_x) < epsilon:
      break
    else:
      results.append([i, x, f_x, f_prime, f_double_prime, x_new, sigma_x])
    x = x_new
  return pd.DataFrame(results, columns=["Iterasi", "x", "f(x)", "f'(x)", "f''(x)", "x_new", "sigma_x"])

In [ ]:
newton_raphson(0, 1e-8, 100)

,Iterasi,x,f(x),f'(x),f''(x),x_new,sigma_x
0,0,0.000000,6.000000e+00,-17.000000,34.000000,0.500000,-4.750000e+00
1,1,0.500000,9.375000e-01,-4.750000,16.000000,0.796875,-1.217361e+00
2,2,0.796875,1.093646e-01,-1.217361,8.151367,0.946219,-2.417755e-01
3,3,0.946219,6.259717e-03,-0.241776,5.002758,0.994548,-2.207658e-02
4,4,0.994548,5.993790e-05,-0.022077,4.098495,0.999934,-2.624036e-04
5,5,0.999934,8.605260e-09,-0.000262,4.001181,1.000000,-3.871069e-08


## Metode Bisection (Bagi dua)

### Algoritma untuk Metode Bisection

**Langkah-langkah:**

1. Pilih dua nilai awal **α_a** dan **α_b** sehingga **αₐ < α_b**  
2. Hitung nilai tengah:
   $$
   \alpha = \alphaₐ + \frac{(\alpha_b - \alphaₐ)}{2}
   $$
3. Evaluasi:
   - Jika **φ(α) = 0**, maka solusi telah konvergen → **selesai**
   - Jika **(α_b - αₐ) < 10^{-4}**, maka toleransi terpenuhi → **selesai**
   - Jika **φ(α) × φ(αₐ) > 0**, maka ganti batas bawah:
     $$
     \alphaₐ \leftarrow \alpha
     $$
   - Jika tidak, ganti batas atas:
     $$
     \alpha_b \leftarrow \alpha
     $$
4. Kembali ke Langkah 2 dan ulangi hingga konvergen.

---

Metode Bisection mencari akar dari fungsi φ(α) dengan mempersempit interval [αₐ, α_b] secara bertahap,  
dengan asumsi bahwa fungsi mengalami perubahan tanda di antara batas bawah dan batas atas.


In [ ]:
def bisection(a, b, epsilon, max_iter):
  results = []
  if turunan_pertama(a)*turunan_pertama(b) > 0:
    print("Fungsi tidak memiliki akar di dalam interval [a, b].")
    return None
  for i in range(max_iter):
    x_tengah = a + (b - a) / 2
    fungsi_tengah = turunan_pertama(x_tengah)
    if fungsi_tengah == 0 or (b - a) < epsilon:
      break
    elif turunan_pertama(a)*fungsi_tengah > 0:
      a = x_tengah
    else:
      b = x_tengah
    results.append([i+1, x_tengah, a, b, fungsi_tengah])
  return pd.DataFrame(results, columns=["Iterasi", "x_tengah", "a", "b", "f(x_tengah)"])

In [ ]:
bisection(0, 4, 1e-4, 100)

,Iterasi,x_tengah,a,b,f(x_tengah)
0,1,2.000000,2.000000,4.000000,-1.000000
1,2,3.000000,2.000000,3.000000,4.000000
2,3,2.500000,2.500000,3.000000,-0.750000
3,4,2.750000,2.500000,2.750000,0.875000
4,5,2.625000,2.625000,2.750000,-0.101562
5,6,2.687500,2.625000,2.687500,0.342773
6,7,2.656250,2.625000,2.656250,0.109985
7,8,2.640625,2.625000,2.640625,0.001602
8,9,2.632812,2.632812,2.640625,-0.050627
9,10,2.636719,2.636719,2.640625,-0.024675


## Golden Section

### Algoritma Golden Section

Deskripsi Singkat
Metode *Golden Section* digunakan untuk mencari **nilai minimum** dari suatu fungsi satu variabel tanpa memerlukan turunan.  
Prinsipnya adalah mempersempit interval pencarian menggunakan perbandingan dua titik yang dibagi berdasarkan rasio emas (*Golden Ratio*).

---

**Step 1 – Inisialisasi**
1. Pilih batas awal:
   $$
   α^{low}, \; α^{up}
   $$
   dengan $$( α^{low} < α^{up} )$$
2. Tentukan:
   $$
   τ = 0.38197
   $$
   (berasal dari *Golden Ratio*).
3. Tentukan toleransi kesalahan:
   $$
   ε = (\Delta α)_{final} / (α^{up} - α^{low})
   $$
4. Tentukan jumlah iterasi:
   $$
   N = -2.078 \ln(ε)
   $$
5. Set \( i = 1 \).

---
**Step 2 – Hitung Titik Awal**
$$
α_1 = (1 - τ) α^{low} + τ α^{up}
$$
$$
α_2 = τ α^{low} + (1 - τ) α^{up}
$$

Hitung nilai fungsi pada kedua titik:
$$
f_1 = f(α_1), \quad f_2 = f(α_2)
$$

---
**Step 3 – Perbandingan Nilai Fungsi**

Selama \( i < N \):
##### Jika \( f_1 > f_2 \):
$$
α^{low} \leftarrow α_1
$$
$$
α_1 \leftarrow α_2, \quad f_1 \leftarrow f_2
$$
$$
α_2 = τ α^{low} + (1 - τ) α^{up}, \quad f_2 = f(α_2)
$$
Naikkan iterasi \( i \leftarrow i + 1 \), lalu kembali ke Step 3.

---

##### Jika \( f_2 > f_1 \):
$$
α^{up} \leftarrow α_2
$$
$$
α_2 \leftarrow α_1, \quad f_2 \leftarrow f_1
$$
$$
α_1 = (1 - τ) α^{low} + τ α^{up}, \quad f_1 = f(α_1)
$$
Naikkan iterasi \( i \leftarrow i + 1 \), lalu kembali ke Step 3.

---

### **Kriteria Berhenti**
Hentikan iterasi jika:
$$
|α^{up} - α^{low}| < ε
$$
atau jumlah iterasi telah mencapai \( N \).

### Kodingan

In [ ]:
def golden_section(a, b, epsilon, max_iter):
  tau = 0.38197
  results = []
  x1 = (1 - tau) * a + tau * b
  x2 = tau * a + (1 - tau) * b
  f1 = fungsi_objektif(x1)
  f2 = fungsi_objektif(x2)
  for i in range(max_iter):
    if abs(b - a) < epsilon:
      break
    elif f1 > f2:
      a = x1
      x1 = x2
      f1 = f2
      x2 = tau * a + (1 - tau) * b
      f2 = fungsi_objektif(x2)
    else:
      b = x2
      x2 = x1
      f2 = f1
      x1 = (1 - tau) * a + tau * b
      f1 = fungsi_objektif(x1)
    results.append([i+1, a, b, f1, f2])
  return pd.DataFrame(results, columns=["Iterasi", "a", "b", "f(a)", "f(b)"])

In [ ]:
golden_section(2, 3, 1e-4, 100)

,Iterasi,a,b,f(a),f(b)
0,1,2.000000,2.618030,-1.252329,-1.103317
1,2,2.000000,2.381970,-1.215518,-1.252329
2,3,2.145901,2.381970,-1.252329,-1.228775
3,4,2.145901,2.291799,-1.248507,-1.252329
4,5,2.201630,2.291799,-1.252329,-1.247834
5,6,2.201630,2.257357,-1.252449,-1.252329
6,7,2.201630,2.236069,-1.251538,-1.252449
7,8,2.214784,2.236069,-1.252449,-1.252638
8,9,2.222916,2.236069,-1.252638,-1.252610
9,10,2.222916,2.231045,-1.252600,-1.252638


## Metode Secant

### Algoritma Metode Secant

1. Inisialisasi batas awal `a`, batas akhir `b`, dan tentukan **toleransi**.
2. Tentukan fungsi objektif `f(x)`.
3. Inisialisasi:
   - `x0` = batas awal
   - `x1` = batas akhir
   - `f0` = `f(x0)`
   - `f1` = `f(x1)`
4. Selama `|x1 - x0| > toleransi`, lakukan langkah-langkah berikut:
   - Hitung titik baru:
$$
x = x_0 - \frac{f_0 (x_0 - x_1)}{f_0 - f_1}
$$
   - Perbarui nilai:
     - `x0 ← x1`
     - `x1 ← x`
     - `f0 ← f1`
     - `f1 ← f(x)`
   - Simpan nilai `x` ke dalam daftar akar.
5. Ulangi hingga konvergen.

In [ ]:
def fungsi_objektif(x):
  return 4*x**3-21*x**2+34*x-17

In [ ]:
def secant_method(a, b, epsilon, max_iter):
    x0 = a
    x1 = b
    results = []

    f0 = fungsi_objektif(x0)
    f1 = fungsi_objektif(x1)

    results.append([0, x0, x1, f0, f1])

    for i in range(max_iter):
        if abs(f1 - f0) < 1e-15:
            print("Error: f1 dan f0 terlalu mirip (potensi pembagian dengan nol).")
            break

        x = x1 - f1 * (x1 - x0) / (f1 - f0)

        if abs(x - x1) < epsilon:
            results.append([i + 1, x1, x, f1, fungsi_objektif(x)])
            print(f"Konvergensi tercapai pada iterasi ke-{i+1}.")
            break

        x0, f0 = x1, f1
        x1, f1 = x, fungsi_objektif(x)

        results.append([i + 1, x0, x1, f0, f1])

    if i == max_iter - 1:
        print("Gagal konvergen setelah", max_iter, "iterasi.")

    return pd.DataFrame(results, columns=["Iterasi", "x0", "x1", "f(x0)", "f(x1)"])

In [ ]:
def secant_method(a, b, epsilon, max_iter):
    """
    Mencari akar persamaan menggunakan Metode Secant.

    Parameters:
    func     : Fungsi objektif f(x)
    a, b     : Tebakan awal x0 dan x1
    epsilon  : Toleransi error
    max_iter : Batas maksimum iterasi
    """

    x0 = a
    x1 = b
    results = []

    f0 = fungsi_objektif(x0)
    f1 = fungsi_objektif(x1)

    # Mencatat kondisi awal (Iterasi 0)
    results.append([0, x0, x1, f0, f1, abs(x1-x0)])

    converged = False

    for i in range(max_iter):
        # Cek pembagian dengan nol
        if abs(f1 - f0) < 1e-15:
            print("Error: f1 dan f0 terlalu mirip (potensi pembagian dengan nol).")
            break

        # Rumus Secant
        x_new = x1 - f1 * (x1 - x0) / (f1 - f0)

        # Hitung error relatif/absolut
        error = abs(x_new - x1)

        # Cek Konvergensi
        if error < epsilon:
            f_new = fungsi_objektif(x_new)
            results.append([i + 1, x1, x_new, f1, f_new, error])
            print(f"Konvergensi tercapai pada iterasi ke-{i+1}. Akar: {x_new:.6f}")
            converged = True
            break

        # Update nilai untuk iterasi berikutnya
        x0, f0 = x1, f1
        x1, f1 = x_new, fungsi_objektif(x_new)

        # Simpan hasil iterasi ini
        results.append([i + 1, x0, x1, f0, f1, error])

    if not converged:
        print(f"Gagal konvergen setelah {max_iter} iterasi.")

    # Kembalikan DataFrame
    return pd.DataFrame(results, columns=["Iterasi", "x_prev", "x_current", "f(x_prev)", "f(x_current)", "Error"])

In [ ]:
def fungsi_objektif(x):
    return 12*x**2 - 42*x + 34

In [ ]:
secant_method(2, 3, 1e-4, 100)

Konvergensi tercapai pada iterasi ke-6. Akar: 2.228714


,Iterasi,x_prev,x_current,f(x_prev),f(x_current),Error
0,0,2.000000,3.000000,-2.000000,1.600000e+01,1.000000
1,1,3.000000,2.111111,16.000000,-1.185185e+00,0.888889
2,2,2.111111,2.172414,-1.185185,-6.087990e-01,0.061303
3,3,2.172414,2.237164,-0.608799,9.794298e-02,0.064750
4,4,2.237164,2.228191,0.097943,-6.006017e-03,0.008973
5,5,2.228191,2.228709,-0.006006,-5.260235e-05,0.000518
6,6,2.228709,2.228714,-0.000053,2.875265e-08,0.000005


## Metode Fibonacci

In [ ]:
import pandas as pd

# 1. Tentukan jumlah bilangan Fibonacci yang diinginkan
n = 15

# 2. Buat deret Fibonacci
fib_list = []
a, b = 1, 1
for _ in range(n):
    fib_list.append(a)
    a, b = b, a + b

# 3. Buat list untuk iterasi (mulai dari 1)
# Kita gunakan range(1, n + 1) agar dimulai dari 1
iter_list = list(range(n))

# 4. Gabungkan kedua list ke dalam dictionary
# Dictionary keys akan menjadi nama kolom
data = {
    'Iterasi': iter_list,
    'Fibonacci': fib_list
}

# 5. Buat DataFrame dari dictionary
list_fib = pd.DataFrame(data)

# 6. Tampilkan DataFrame
print(list_fib)

    Iterasi  Fibonacci
0         0          1
1         1          1
2         2          2
3         3          3
4         4          5
5         5          8
6         6         13
7         7         21
8         8         34
9         9         55
10       10         89
11       11        144
12       12        233
13       13        377
14       14        610


In [ ]:
#Cek N+1
target_name = 8

# Select the row where 'Name' column matches the target_name
ambil_angka = list_fib.loc[list_fib['Fibonacci'] >= target_name]
angka_terakhir = ambil_angka['Iterasi'].iloc[0]

print(angka_terakhir)

5


In [ ]:
def fungsi_objektif(x):
  return x**4-14*x**3+60*x**2-70*x

In [ ]:
def fibonacci_method(a, b, n):
    # n adalah jumlah iterasi (jumlah bilangan fibonacci yang digunakan dikurangi 1)
    # f_nplus1 = fib_list[n]
    # f_nplus2 = fib_list[n+1]

    # Jika n melebihi panjang list fibonacci, hitung ulang deret fibonacci hingga n+2
    if n >= len(fib_list):
      fib_list_extended = []
      a_fib, b_fib = 1, 1
      for _ in range(n + 2):
        fib_list_extended.append(a_fib)
        a_fib, b_fib = b_fib, a_fib + b_fib
      f_nplus1 = fib_list_extended[n]
      f_nplus2 = fib_list_extended[n + 1]
    else:
      f_nplus1 = fib_list[n]
      f_nplus2 = fib_list[n+1]

    # Hitung nilai awal alpha1 dan alpha2
    alpha1 = a + (f_nplus1 / f_nplus2) * (b - a)
    alpha2 = a + (1 - (f_nplus1 / f_nplus2)) * (b - a)

    # Hitung nilai fungsi pada alpha1 dan alpha2
    f_alpha1 = fungsi_objektif(alpha1)
    f_alpha2 = fungsi_objektif(alpha2)

    results = []
    results.append([0, a, b, alpha1, alpha2, f_alpha1, f_alpha2])

    for k in range(1, n):
        # f_nplus1_minus_k = fib_list[n - k]
        # f_nplus2_minus_k = fib_list[n - k + 1]

        # Jika n-k atau n-k+1 melebihi panjang list fibonacci, hitung ulang
        if n - k >= len(fib_list):
          fib_list_extended = []
          a_fib, b_fib = 1, 1
          for _ in range(n - k + 2):
            fib_list_extended.append(a_fib)
            a_fib, b_fib = b_fib, a_fib + b_fib
          f_nplus1_minus_k = fib_list_extended[n - k]
          f_nplus2_minus_k = fib_list_extended[n - k + 1]
        else:
          f_nplus1_minus_k = fib_list[n - k]
          f_nplus2_minus_k = fib_list[n - k + 1]


        if f_alpha1 < f_alpha2:
            b = alpha2
            alpha2 = alpha1
            f_alpha2 = f_alpha1
            alpha1 = a + (f_nplus1_minus_k / f_nplus2_minus_k) * (b - a)
            f_alpha1 = fungsi_objektif(alpha1)
        else:
            a = alpha1
            alpha1 = alpha2
            f_alpha1 = f_alpha2
            alpha2 = a + (1 - (f_nplus1_minus_k / f_nplus2_minus_k)) * (b - a)
            f_alpha2 = fungsi_objektif(alpha2)
        results.append([k, a, b, alpha1, alpha2, f_alpha1, f_alpha2])

    return pd.DataFrame(results, columns=["Iterasi", "a", "b", "alpha1", "alpha2", "f(alpha1)", "f(alpha2)"])

In [ ]:
fibonacci_method(0, 2, 4)

,Iterasi,a,b,alpha1,alpha2,f(alpha1),f(alpha2)
0,0,0.00,2.00,1.25,0.75,-18.652344,-24.339844
1,1,1.25,2.00,0.75,1.55,-24.339844,-10.712244
2,2,1.25,1.55,1.45,0.75,-13.610244,-24.339844
3,3,1.45,1.55,0.75,1.50,-24.339844,-12.187500


In [ ]:
import numpy as np
import pandas as pd

#definisikan fungsi objektif nya
def fungsi_objektif(x):
  return x**2+4*np.cos(x)

#definisikan turunan pertama fungsi objektif
def turunan_pertama(x):
  return 2*x-4*np.sin(x)

#definisikan turunan kedua fungsi obkektif
def turunan_kedua(x):
  return 2-4*np.cos(x)

In [ ]:
def newton_raphson(x0, epsilon, max_iter):
  x = x0
  results = []
  for i in range(max_iter):
    f_x = fungsi_objektif(x)
    f_prime = turunan_pertama(x)
    f_double_prime = turunan_kedua(x)
    delta_x = -f_prime / f_double_prime
    x_new = x + delta_x
    sigma_x = turunan_pertama(x_new)
    if abs(x_new-x) < epsilon:
      results.append([i, x, f_x, f_prime, f_double_prime, x_new, sigma_x])
      break
    else:
      results.append([i, x, f_x, f_prime, f_double_prime, x_new, sigma_x])
    x = x_new
  return pd.DataFrame(results, columns=["Iterasi", "x", "f(x)", "f'(x)", "f''(x)", "x_new", "sigma_x"])

In [ ]:
newton_raphson(1.5, 0.05, 100)

,Iterasi,x,f(x),f'(x),f''(x),x_new,sigma_x
0,0,1.500000,2.532949,-0.989980,1.717051,2.076558,0.653894
1,1,2.076558,2.374198,0.653894,3.937896,1.910507,0.049608
2,2,1.910507,2.317180,0.049608,3.332856,1.895622,0.000419
